In [ ]:
# Setup
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()
client = OpenAI(api_key=os.getenv('OPENAI_API_KEY'))

def ask(prompt, model="gpt-3.5-turbo", temperature=0.7):
    """Helper to call LLM."""
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=temperature
    )
    return response.choices[0].message.content

## 1. Zero-Shot Prompting

No examples provided - rely on model's pre-training.

In [ ]:
# Zero-shot: Direct question
prompt = "What is the capital of France?"
print(ask(prompt))

# Zero-shot: Complex task
prompt = """
Classify the sentiment of this review as positive, negative, or neutral:

Review: "The product works okay but the customer service was terrible."

Sentiment:
"""
print(ask(prompt, temperature=0))

## 2. One-Shot Prompting

Provide one example to guide the model.

In [ ]:
# One-shot: Show format
prompt = """
Convert product descriptions to JSON format.

Example:
Input: "iPhone 15 Pro in blue, 256GB storage, $999"
Output: {"product": "iPhone 15 Pro", "color": "blue", "storage": "256GB", "price": 999}

Now convert this:
Input: "Samsung Galaxy S24 in black, 512GB storage, $1199"
Output:
"""
print(ask(prompt, temperature=0))

## 3. Few-Shot Prompting

Multiple examples = better results. The gold standard!

In [ ]:
# Few-shot: Classification task
prompt = """
Classify customer feedback as Bug, Feature Request, or Question.

Example 1:
Feedback: "The app crashes when I upload large files"
Category: Bug

Example 2:
Feedback: "Can you add dark mode?"
Category: Feature Request

Example 3:
Feedback: "How do I export my data?"
Category: Question

Now classify:
Feedback: "It would be great to have keyboard shortcuts"
Category:
"""
print(ask(prompt, temperature=0))

In [ ]:
# Few-shot: Code generation with style
prompt = """
Write Python functions following this style:

Example 1:
Task: Calculate factorial
def factorial(n: int) -> int:
    '''Calculate factorial of n.
    
    Args:
        n: Non-negative integer
    
    Returns:
        Factorial of n
    '''
    if n <= 1:
        return 1
    return n * factorial(n - 1)

Example 2:
Task: Check if prime
def is_prime(n: int) -> bool:
    '''Check if number is prime.
    
    Args:
        n: Integer to check
    
    Returns:
        True if prime, False otherwise
    '''
    if n < 2:
        return False
    for i in range(2, int(n ** 0.5) + 1):
        if n % i == 0:
            return False
    return True

Now write:
Task: Calculate fibonacci number
"""
print(ask(prompt, temperature=0))

## 4. Dynamic Few-Shot Selection

Choose examples based on similarity to the query.

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# Example pool
examples = [
    {"input": "The food was delicious!", "output": "positive"},
    {"input": "Terrible service, never coming back", "output": "negative"},
    {"input": "It was okay, nothing special", "output": "neutral"},
    {"input": "Best restaurant in town!", "output": "positive"},
    {"input": "Worst experience ever", "output": "negative"},
]

def get_embeddings(texts):
    """Get embeddings for similarity."""
    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=texts
    )
    return [item.embedding for item in response.data]

def select_examples(query, examples, k=2):
    """Select k most similar examples."""
    # Get embeddings
    all_texts = [query] + [ex["input"] for ex in examples]
    embeddings = get_embeddings(all_texts)
    
    query_emb = np.array(embeddings[0]).reshape(1, -1)
    example_embs = np.array(embeddings[1:])
    
    # Compute similarities
    similarities = cosine_similarity(query_emb, example_embs)[0]
    
    # Get top k
    top_indices = similarities.argsort()[-k:][::-1]
    return [examples[i] for i in top_indices]

# Test
query = "Amazing food and great atmosphere!"
selected = select_examples(query, examples, k=2)

# Build prompt with selected examples
prompt = "Classify sentiment:\n\n"
for ex in selected:
    prompt += f"Input: {ex['input']}\nOutput: {ex['output']}\n\n"
prompt += f"Input: {query}\nOutput:"

print("Selected examples:", [ex["input"] for ex in selected])
print("\nResult:", ask(prompt, temperature=0))

## 5. Best Practices

### ✅ DO
- Use few-shot for consistency
- Provide diverse examples
- Show edge cases
- Use clear delimiters
- Set temperature=0 for consistency

### ❌ DON'T
- Use conflicting examples
- Provide too many examples (>10)
- Forget to test edge cases
- Use vague examples

## Exercise: Build Your Own Classifier

Create a few-shot prompt to classify programming language from code snippets.

In [ ]:
# Your turn!
prompt = """
Identify the programming language:

Example 1:
Code: def hello(): print("Hello")
Language: Python

# Add 2-3 more examples
# ...

Now identify:
Code: const greeting = () => console.log("Hello");
Language:
"""

# Test your prompt
# print(ask(prompt, temperature=0))

## Key Takeaways

1. **Zero-shot**: Quick, works for simple tasks
2. **One-shot**: Good for showing format
3. **Few-shot**: Best for consistency and complex tasks
4. **Dynamic selection**: Choose relevant examples automatically
5. **Quality > Quantity**: 3-5 good examples beat 20 mediocre ones

## Next Steps

- `02_chain_of_thought.ipynb` - Make models reason step-by-step
- `03_react_prompting.ipynb` - Combine reasoning with actions
- `05_prompt_templates.ipynb` - Build reusable prompt systems